# MRCP: ResNet-50 training

Train a binary classifier for CBD stones. Set the Drive paths below, then run the cells in order. The default paths use non-cropped images.


In [ ]:
%pip install -q torch torchvision scikit-learn numpy matplotlib pillow


In [ ]:
import os
import random
import copy
import json

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc,
    precision_recall_curve,
    classification_report
)
from sklearn.model_selection import StratifiedShuffleSplit

from google.colab import drive


In [ ]:
# -----------------------
# 0. Device & seed
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# -----------------------
# 1. Mount Drive & paths
# -----------------------
drive.mount('/content/drive')

# CHANGE THIS if your folder is different:
BASE_DIR = "/content/drive/MyDrive/mrcp_data/non_cropped"
INTERNAL_DIR = os.path.join(BASE_DIR, "internal")
EXTERNAL_DIR = os.path.join(BASE_DIR, "external")

assert os.path.isdir(INTERNAL_DIR), f"Internal dir not found: {INTERNAL_DIR}"
assert os.path.isdir(EXTERNAL_DIR), f"External dir not found: {EXTERNAL_DIR}"

print("Internal data directory:", INTERNAL_DIR)
print("External data directory:", EXTERNAL_DIR)

# Output directory for model, metrics, plots
OUTPUT_DIR = "/content/drive/MyDrive/mrcp_outputs/non_cropped"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_DIR)


In [ ]:
# -----------------------
# 2. Transforms & dataset helper
# -----------------------
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class SubsetWithTransform(Dataset):
    """Subset of a dataset with its own transform."""
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.samples = [self.base_dataset.samples[i] for i in self.indices]
        self.targets = [self.base_dataset.targets[i] for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self.base_dataset.loader(path)
        if self.transform is not None:
            img = self.transform(img)
        return img, label


In [ ]:
# -----------------------
# 3. Internal dataset & stratified 70/20/10 split
# -----------------------
internal_base = datasets.ImageFolder(INTERNAL_DIR)
class_names = internal_base.classes
print("Detected classes:", class_names)

# Class 1 is the positive class used by the model and metrics.
if internal_base.class_to_idx != {"0_no_stone": 0, "1_CBD_stone": 1}:
    raise ValueError("Expected class folders: 0_no_stone and 1_CBD_stone.")

num_images = len(internal_base)
print("Total internal images:", num_images)

all_labels = np.array(internal_base.targets)
all_indices = np.arange(num_images)

# First split: train (70%) vs temp (30%)
sss1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)
train_idx, temp_idx = next(sss1.split(all_indices, all_labels))

# Second split: temp -> val (20%) and test (10%) of total
temp_labels = all_labels[temp_idx]
temp_indices_for_split = np.arange(len(temp_idx))

sss2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=1/3,   # 1/3 of 30% = 10% total
    random_state=SEED
)
val_sub_idx, test_sub_idx = next(sss2.split(temp_indices_for_split, temp_labels))

val_idx = temp_idx[val_sub_idx]
test_idx = temp_idx[test_sub_idx]

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

train_dataset = SubsetWithTransform(internal_base, train_idx, transform=train_transform)
val_dataset = SubsetWithTransform(internal_base, val_idx, transform=eval_transform)
test_dataset = SubsetWithTransform(internal_base, test_idx,  transform=eval_transform)

# External dataset (no split)
external_dataset = datasets.ImageFolder(EXTERNAL_DIR, transform=eval_transform)
if external_dataset.class_to_idx != internal_base.class_to_idx:
    raise ValueError("Internal and external class folders must match.")
print("External images:", len(external_dataset))


In [ ]:
# -----------------------
# 4. Dataloaders
# -----------------------
BATCH_SIZE = 32  # adjust if you hit OOM
NUM_WORKERS = 2  # set to 0 if you get dataloader issues in Colab

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
external_loader = DataLoader(external_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))


In [ ]:
# -----------------------
# 5. Model: ResNet-50
# -----------------------
def build_model():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 1)  # single logit for binary
    return model

model = build_model().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Optional: LR scheduler
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=7,
    gamma=0.1
)


In [ ]:
# -----------------------
# 6. Training function
# -----------------------
def train_model(model, criterion, optimizer,
                train_loader, val_loader,
                device, num_epochs=20, scheduler=None):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # ---- Train ----
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.float().to(device)

            optimizer.zero_grad()

            outputs = model(inputs).view(-1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).long()

            running_loss += loss.item() * labels.size(0)
            running_corrects += torch.sum(preds == labels.long()).item()
            total += labels.size(0)

        epoch_train_loss = running_loss / total
        epoch_train_acc = running_corrects / total
        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f}")

        # ---- Val ----
        model.eval()
        val_running_loss = 0.0
        val_running_corrects = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.float().to(device)

                outputs = model(inputs).view(-1)
                loss = criterion(outputs, labels)

                probs = torch.sigmoid(outputs)
                preds = (probs >= 0.5).long()

                val_running_loss += loss.item() * labels.size(0)
                val_running_corrects += torch.sum(preds == labels.long()).item()
                val_total += labels.size(0)

        epoch_val_loss = val_running_loss / val_total
        epoch_val_acc = val_running_corrects / val_total
        print(f"Val   Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        if scheduler is not None:
            scheduler.step()

        # Save best model (by val accuracy)
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print("  -> New best model saved.")

    print(f"\nTraining complete. Best val Acc: {best_val_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model


In [ ]:
# -----------------------
# 7. Evaluation + plots (with saving)
# -----------------------
def evaluate_and_plot(model, dataloader, device,
                      dataset_name="Dataset",
                      class_names=None,
                      save_prefix=None):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            outputs = model(inputs).view(-1)
            probs = torch.sigmoid(outputs)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    if np.unique(all_labels).size != 2:
        raise ValueError(f"{dataset_name} must contain both classes for ROC evaluation.")
    preds = (all_probs >= 0.5).astype(int)

    # ---- Metrics ----
    acc = accuracy_score(all_labels, preds)
    cm = confusion_matrix(all_labels, preds, labels=[0, 1])

    print(f"\n=== {dataset_name} Metrics ===")
    print(f"Accuracy: {acc:.4f}")
    print("\nConfusion matrix (rows = true, cols = predicted):")
    print(cm)

    if class_names is not None and len(class_names) == 2:
        print("\nClass order:", class_names)
    print("\nClassification report:")
    print(classification_report(all_labels, preds, labels=[0, 1], digits=3, zero_division=0))

    # ---- Confusion matrix plot ----
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"{dataset_name} - Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(2)
    if class_names is None:
        class_names = ["Class 0", "Class 1"]
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # ---- ROC curve + AUC ----
    fpr, tpr, roc_thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle=':', label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate (Recall)")
    plt.title(f"{dataset_name} - ROC Curve")
    plt.legend(loc="lower right")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_roc.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # ---- Precision-Recall + AUC ----
    precision, recall, pr_thresholds = precision_recall_curve(all_labels, all_probs)
    pr_auc = auc(recall, precision)

    plt.figure()
    plt.plot(recall, precision, label=f"PR (AUC = {pr_auc:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{dataset_name} - Precision-Recall Curve")
    plt.legend(loc="best")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_pr.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # ---- Exploratory F1 vs threshold ----
    # Use the validation-threshold notebook for a cutoff fixed before test evaluation.
    f1_scores = []
    for p, r in zip(precision[:-1], recall[:-1]):  # thresholds length = len(precision)-1
        if p + r == 0:
            f1_scores.append(0.0)
        else:
            f1_scores.append(2 * p * r / (p + r))
    f1_scores = np.array(f1_scores)

    best_idx = np.argmax(f1_scores)
    best_threshold = pr_thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    plt.figure()
    plt.plot(pr_thresholds, f1_scores)
    plt.xlabel("Threshold")
    plt.ylabel("F1 score")
    plt.title(f"{dataset_name} - F1 vs Threshold")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_f1_threshold.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"\nExploratory best F1 on {dataset_name}: {best_f1:.3f} at threshold {best_threshold:.3f}")

    return {
        "accuracy": float(acc),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "best_f1": float(best_f1),
        "best_threshold": float(best_threshold)
    }


In [ ]:
# -----------------------
# 8. Train & evaluate
# -----------------------
NUM_EPOCHS = 20

print("\nStarting training...")
model = train_model(
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    device,
    num_epochs=NUM_EPOCHS,
    scheduler=scheduler
)

# Save best model weights
model_path = os.path.join(OUTPUT_DIR, "mrcp_resnet50.pth")
torch.save(model.state_dict(), model_path)
print("Saved model weights to:", model_path)

print("\nEvaluating on INTERNAL TEST SET...")
internal_metrics = evaluate_and_plot(
    model,
    test_loader,
    device,
    dataset_name="Internal Test",
    class_names=class_names,
    save_prefix=os.path.join(OUTPUT_DIR, "internal")
)

print("\nEvaluating on EXTERNAL TEST SET...")
external_metrics = evaluate_and_plot(
    model,
    external_loader,
    device,
    dataset_name="External Test",
    class_names=external_dataset.classes,
    save_prefix=os.path.join(OUTPUT_DIR, "external")
)

# Save metrics to JSON
metrics = {
    "internal": internal_metrics,
    "external": external_metrics
}
metrics_path = os.path.join(OUTPUT_DIR, "mrcp_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)
print("Saved metrics to:", metrics_path)

print("\nSummary:")
print("Internal test metrics:", internal_metrics)
print("External test metrics:", external_metrics)
print("\nAll plots, model, and metrics saved to:", OUTPUT_DIR)
